<a href="https://colab.research.google.com/github/miaouiAmine/NLP-Translation-Demo/blob/main/Attention_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Step 1: Setup and Data Download
!wget https://www.manythings.org/anki/fra-eng.zip
!unzip fra-eng.zip

--2025-02-12 21:17:14--  https://www.manythings.org/anki/fra-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7943074 (7.6M) [application/zip]
Saving to: ‘fra-eng.zip’

fra-eng.zip         100%[===================>]   7.57M  10.1MB/s    in 0.7s    

2025-02-12 21:17:15 (10.1 MB/s) - ‘fra-eng.zip’ saved [7943074/7943074]

Archive:  fra-eng.zip
  inflating: _about.txt              
  inflating: fra.txt                 


In [21]:
# Step 2: Imports
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Attention, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Bidirectional,
    Attention, Concatenate, Embedding
)
from tensorflow.keras.models import Model

In [39]:
# Step 3: Data Preparation
def load_data(path, num_samples=30000):
    with open(path) as f:
        lines = f.read().split('\n')[:-1]

    pairs = [line.split('\t')[:2] for line in lines[:num_samples]]
    eng_texts, fr_texts = zip(*pairs)
    # Basic cleaning
    eng_texts = [text.lower().replace('\u202f', ' ').replace('—', ' ') for text in eng_texts]
    fr_texts = ['<start> ' + text.lower().replace('\u202f', ' ').replace('—', ' ') + ' <end>'
                for text in fr_texts]

    return eng_texts, fr_texts

eng_texts, fr_texts = load_data('fra.txt', num_samples=30000)

In [43]:
print("--",eng_texts[0], fr_texts[0],"--")
print("--",eng_texts[-1], fr_texts[-1],"--")

-- go. <start> va ! <end> --
-- tom felt betrayed. <start> tom se sentait trahi. <end> --


In [41]:
# Step 4: Text Vectorization
max_vocab_size = 15000
max_sequence_length = 20

# English vectorizer
eng_vectorizer = TextVectorization(
    max_tokens=max_vocab_size,
    standardize='lower_and_strip_punctuation',
    output_mode='int',
    output_sequence_length=max_sequence_length
)

# French vectorizer
fr_vectorizer = TextVectorization(
    max_tokens=max_vocab_size,
    standardize='lower_and_strip_punctuation',
    output_mode='int',
    output_sequence_length=max_sequence_length + 1  # +1 for offset
)

# Adapt vectorizers
eng_vectorizer.adapt(eng_texts)
fr_vectorizer.adapt(['<start> ' + t + ' <end>' for t in fr_texts])

In [16]:
# Step 5: Dataset Pipeline
def format_dataset(eng, fr):
    eng = eng_vectorizer(eng)
    fr_in = fr_vectorizer(fr)[:, :-1]
    fr_out = fr_vectorizer(fr)[:, 1:]
    return (eng, fr_in), fr_out

def make_dataset(eng_texts, fr_texts, batch_size=64):
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, fr_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset)
    return dataset.cache().prefetch(tf.data.AUTOTUNE)

# Train/val split
split = int(0.8 * len(eng_texts))
train_ds = make_dataset(eng_texts[:split], fr_texts[:split])
val_ds = make_dataset(eng_texts[split:], fr_texts[split:])

In [17]:
# Step 6: Model Architecture with Attention
embed_dim = 256
latent_dim = 512

# Encoder
encoder_inputs = tf.keras.Input(shape=(None,), name='encoder_inputs')
enc_emb = Embedding(input_dim=eng_vectorizer.vocabulary_size(),
                    output_dim=embed_dim)(encoder_inputs)
encoder_lstm = Bidirectional(LSTM(latent_dim, return_sequences=True, return_state=True))
encoder_outputs, fw_h, fw_c, bw_h, bw_c = encoder_lstm(enc_emb)
state_h = Concatenate()([fw_h, bw_h])
state_c = Concatenate()([fw_c, bw_c])
encoder_states = [state_h, state_c]

# Decoder with Attention
decoder_inputs = tf.keras.Input(shape=(None,), name='decoder_inputs')
dec_emb = Embedding(input_dim=fr_vectorizer.vocabulary_size(),
                    output_dim=embed_dim)(decoder_inputs)
decoder_lstm = LSTM(2*latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# Attention layer
attention = Attention()([decoder_outputs, encoder_outputs])
decoder_concat = Concatenate()([decoder_outputs, attention])

# Dense output
decoder_dense = Dense(fr_vectorizer.vocabulary_size(), activation='softmax')
outputs = decoder_dense(decoder_concat)

# Full model
model = Model([encoder_inputs, decoder_inputs], outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [10]:
# Step 7: Training
checkpoint = ModelCheckpoint('transformer_model.keras', save_best_only=True)
early_stop = EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 84s 209ms/step - accuracy: 0.8255 - loss: 1.5176 - val_accuracy: 0.8144 - val_loss: 1.2798
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 82s 218ms/step - accuracy: 0.8702 - loss: 0.8682 - val_accuracy: 0.8149 - val_loss: 1.2185
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 82s 218ms/step - accuracy: 0.8805 - loss: 0.7470 - val_accuracy: 0.8313 - val_loss: 1.1548
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 82s 218ms/step - accuracy: 0.8932 - loss: 0.6227 - val_accuracy: 0.8490 - val_loss: 1.0642
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 82s 218ms/step - accuracy: 0.9043 - loss: 0.5094 - val_accuracy: 0.8569 - val_loss: 1.0109
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 82s 219ms/step - accuracy: 0.9139 - loss: 0.4104 - val_accuracy: 0.8610 - val_loss: 0.9790
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 82s 218ms/step - accuracy: 0.9248 - loss: 0.3278 - val_accuracy: 0.8685 - val_loss: 0.9401
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 76s 202ms/step - accuracy: 0.9348 - loss: 0

In [22]:
# Step 8: Inference Model
encoder_model = Model(encoder_inputs, [encoder_outputs, encoder_states])

decoder_state_input_h = Input(shape=(2*latent_dim,), name='decoder_state_input_h')
decoder_state_input_c = Input(shape=(2*latent_dim,), name='decoder_state_input_c')
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, dh, dc = decoder_lstm(
    dec_emb, initial_state=decoder_states_inputs
)
attention_inf = Attention()([decoder_outputs, encoder_outputs])
decoder_concat_inf = Concatenate()([decoder_outputs, attention_inf])
decoder_outputs = decoder_dense(decoder_concat_inf)

decoder_model = Model(
    [decoder_inputs] + [encoder_outputs] + decoder_states_inputs,
    [decoder_outputs] + [dh, dc]
)

In [25]:
# Step 9: Translation Function
def translate(input_text, temperature=1.0):
    # Preprocess input
    input_text = input_text.lower().strip()
    tokenized_input = eng_vectorizer([input_text])

    # Encode input
    enc_out, states = encoder_model.predict(tokenized_input)

    # Generate empty target sequence
    decoded_tokens = ['<start>']
    current_token = '<start>'

    for _ in range(max_sequence_length):
        tokenized_target = fr_vectorizer([' '.join(decoded_tokens)])
        target_seq = tokenized_target[:, :-1]

        preds, h, c = decoder_model.predict(
            [target_seq] + [enc_out] + states,
            verbose=0
        )

        # Sample next token
        sampled_token_index = tf.random.categorical(preds[:, -1, :]/temperature, 1)[0, 0].numpy()
        sampled_token = fr_vectorizer.get_vocabulary()[sampled_token_index]

        if sampled_token == '<end>' or len(decoded_tokens) >= max_sequence_length:
            break

        decoded_tokens.append(sampled_token)
        states = [h, c]

    # Remove start token and join
    return ' '.join(decoded_tokens[1:]).replace(' <end>', '')

In [45]:
# Step 10: Test Translations
test_phrases = [
    "go",
    "tom felt betrayed"
]

for phrase in test_phrases:
    translation = translate(phrase)
    print(f"English: {phrase}")
    print(f"French: {translation}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
English: go
French: abandonnèrent parie acides traitement cloîtrée chapeau  sembler compteur déverrouillele lépeler très imitation taimait varier cesser menteuses démarrez aidonsle tueur

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
English: tom felt betrayed
French: rejouele sécoulait devoirs encombre meurent épousa téléphoné désert lencre lai grandiront vos lastrologie botte nettoyé illuminé tailleur rigolez  correctement

